# File 1: Weighted Early Fusion — Learn Modality Weights

**Goal:** Instead of simple concatenation (equal contribution), learn optimal per-modality
scalar weights using all 14 EvoloPy metaheuristics.

**Search space:** 4 continuous values in [0, 1] — one weight per modality (depth, sensor, skeleton, video).

**Pipeline per fold:**
1. Metaheuristic optimizes `[w_depth, w_sensor, w_skeleton, w_video]` using quick fitness evaluations
2. Apply learned weights → train full classifier → evaluate

**Outputs saved:**
- Best weights per fold (per optimizer)
- Best-overall weights per fold (best optimizer chosen per fold by val accuracy)
- Baseline (unweighted) results for comparison
- All results CSVs, plots, and a `best_weights_per_fold.pkl` for File 2

In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [12]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# EvoloPy imports
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch version: 2.9.1+cu128
GPU: NVIDIA GeForce RTX 4090


## Configuration

In [13]:
# Paths
FEATURE_DIR = Path("features_new")

RESULTS_ROOT = Path("results_weighted_fusion")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Training hyperparameters (same as original)
N_FOLDS = 5
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Metaheuristic hyperparameters
N_POPULATION = 20
MAX_ITERATIONS = 30
FITNESS_EPOCHS = 15  # quick training during fitness evaluation

# All 14 EvoloPy optimizers
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline'] + [f'weighted_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  Fitness epochs: {FITNESS_EPOCHS}")
print(f"  Optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")

Configuration:
  N_FOLDS: 5
  N_EPOCHS: 300
  Metaheuristic pop: 20, iter: 30
  Fitness epochs: 15
  Optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 15


## Load Data

In [14]:
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")

print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")

first_sample = X_feat[0]
FEATURE_DIMS = {
    'depth': first_sample['depth_feat'].shape[0],
    'sensor': first_sample['sensor_feat'].shape[0],
    'skeleton': first_sample['skeleton_feat'].shape[0]
}
MODALITY_NAMES = list(FEATURE_DIMS.keys())
MODALITY_DIMS = list(FEATURE_DIMS.values())
TOTAL_FEATURES = sum(MODALITY_DIMS)
N_MODALITIES = len(MODALITY_NAMES)

print(f"\nFeature dimensions:")
for modality, dim in FEATURE_DIMS.items():
    print(f"  {modality}: {dim}")
print(f"  TOTAL: {TOTAL_FEATURES}")

Loaded 1165 samples
Number of classes: 22
Number of subjects: 5

Feature dimensions:
  depth: 216
  sensor: 652
  skeleton: 1645
  TOTAL: 2513


## Model & Dataset

In [15]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (same as original)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)


print("Model and dataset defined")

Model and dataset defined


## Helper Functions

In [16]:
def prepare_unified_features(X_feat_list):
    """Concatenate all modality features into unified vector (no weighting)"""
    unified = []
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['sensor_feat'],
            sample['skeleton_feat']
        ])
        unified.append(feat_vector)
    return np.array(unified)


def apply_modality_weights(X, weights):
    """Apply per-modality scalar weights to concatenated feature matrix.
    
    Args:
        X: (N, TOTAL_FEATURES) — already concatenated and scaled
        weights: (N_MODALITIES,) — one weight per modality
    Returns:
        Weighted copy of X
    """
    X_weighted = X.copy()
    start_idx = 0
    for i, dim in enumerate(MODALITY_DIMS):
        end_idx = start_idx + dim
        X_weighted[:, start_idx:end_idx] *= weights[i]
        start_idx = end_idx
    return X_weighted


def count_model_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    return X.nbytes / (1024 ** 2)


print("Helper functions defined")

Helper functions defined


## Fitness Function & Quick Training

In [17]:
def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation.
    Returns val_loss (continuous) and val_acc.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc


def create_fitness_modality_weights(X_train, y_train, X_val, y_val, num_classes):
    """Fitness function for optimizing 4 modality weights.
    
    Uses VALIDATION LOSS (continuous) instead of (1 - accuracy).
    
    Why: With only 4 dimensions and ~107 val samples, many weight
    combinations achieve 100% val accuracy -> fitness=0.0 -> optimizer
    cannot differentiate. Val loss is continuous and always provides
    a gradient signal even when accuracy is saturated.
    """
    eval_count = [0]
    
    def fitness(solution):
        try:
            weights = np.clip(solution, 0.0, 1.0)
            
            # Apply modality weights
            X_tr_w = apply_modality_weights(X_train, weights)
            X_val_w = apply_modality_weights(X_val, weights)
            
            train_ds = MultiModalDataset(X_tr_w, y_train)
            val_ds = MultiModalDataset(X_val_w, y_val)
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
            
            model = SimpleNN(TOTAL_FEATURES, num_classes).to(DEVICE)
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=FITNESS_EPOCHS, lr=1e-3, device=DEVICE
            )
            
            del model, train_ds, val_ds, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            # Use val_loss directly as fitness (continuous, always differentiable)
            # Even when accuracy=100%, loss still varies between solutions
            fitness_val = val_loss
            return fitness_val
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 100.0
    
    return fitness


def run_weight_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes):
    """Run a single metaheuristic to optimize modality weights."""
    print(f"    Running weighted_{optimizer_name}...")
    print(f"      Search dim: {N_MODALITIES}, Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    fitness_func = create_fitness_modality_weights(X_train, y_train, X_val, y_val, num_classes)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, N_MODALITIES, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    learned_weights = np.clip(solution.bestIndividual, 0.0, 1.0)
    convergence = (solution.convergence.tolist()
                   if hasattr(solution.convergence, 'tolist')
                   else list(solution.convergence))
    
    results = {
        'modality_weights': learned_weights,
        'convergence': convergence,
        'best_fitness': float(convergence[-1]) if len(convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'method': f'weighted_{optimizer_name}'
    }
    
    w_str = ', '.join([f"{MODALITY_NAMES[i]}={learned_weights[i]:.3f}" for i in range(N_MODALITIES)])
    print(f"      Weights: {w_str}")
    print(f"      Best fitness: {results['best_fitness']:.6f}, Time: {exec_time:.1f}s")
    
    return results


print("Fitness function and weight optimizer runner defined")
print("NOTE: Using val_loss as fitness (continuous signal, not accuracy)")


Fitness function and weight optimizer runner defined
NOTE: Using val_loss as fitness (continuous signal, not accuracy)


## Full Training & Evaluation

In [18]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics (same as original)"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_val_loss = float('inf')
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)
        epoch_val_acc = val_correct / val_total

        # Also track val loss for this epoch
        epoch_val_loss = 0.0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                labels = labels.to(DEVICE)
                outputs = model(features)
                epoch_val_loss += nn.CrossEntropyLoss()(outputs, labels).item()
        epoch_val_loss /= len(val_loader)

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_val_loss = epoch_val_loss
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    model.load_state_dict(best_model_state)
    
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'val_loss': best_val_loss,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics


print("Full train/eval function defined")

Full train/eval function defined


## Main Experiment Runner

In [19]:
def run_weighted_fusion_all_folds():
    """
    Run baseline + 14 weighted metaheuristics across all 8 LOSO folds.
    """
    print("=" * 80)
    print("WEIGHTED EARLY FUSION: MODALITY WEIGHT OPTIMIZATION")
    print(f"Methods: baseline + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print(f"Folds: {N_FOLDS}")
    print("=" * 80)
    
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    master_results = {m: {} for m in ALL_METHODS}
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X_unified, y, groups=subjects)):
        print(f"\n{'#' * 80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#' * 80}")
        
        # Split
        X_train_val = X_unified[train_val_idx]
        y_train_val = y[train_val_idx]
        X_test = X_unified[test_idx]
        y_test = y[test_idx]
        
        # Reserve 1 subject for validation (same as original)
        train_val_subjects = np.unique(subjects[train_val_idx])
        val_subject = train_val_subjects[-1]
        val_mask = subjects[train_val_idx] == val_subject
        train_mask = ~val_mask
        
        X_train, y_train = X_train_val[train_mask], y_train_val[train_mask]
        X_val, y_val = X_train_val[val_mask], y_train_val[val_mask]
        
        print(f"  Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Normalize
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)
        
        # =================================================================
        # Run each method
        # =================================================================
        for method_name in ALL_METHODS:
            print(f"\n  --- {method_name.upper()} ---")
            
            fs_start_time = time.time()
            
            if method_name == 'baseline':
                # No weighting — simple concatenation
                modality_weights = np.ones(N_MODALITIES)
                weight_results = {
                    'modality_weights': modality_weights,
                    'execution_time': 0,
                    'convergence': None,
                    'best_fitness': None,
                    'method': 'baseline'
                }
            else:
                opt_name = method_name.replace('weighted_', '')
                opt_func = EVOLOPY_OPTIMIZERS[opt_name]
                weight_results = run_weight_optimizer(
                    opt_name, opt_func, X_train, y_train, X_val, y_val, num_classes
                )
                modality_weights = weight_results['modality_weights']
            
            fs_time = time.time() - fs_start_time
            
            # Apply weights
            X_train_w = apply_modality_weights(X_train, modality_weights)
            X_val_w = apply_modality_weights(X_val, modality_weights)
            X_test_w = apply_modality_weights(X_test, modality_weights)
            
            # Build and train final model
            model = SimpleNN(TOTAL_FEATURES, num_classes).to(DEVICE)
            
            train_dataset = MultiModalDataset(X_train_w, y_train)
            val_dataset = MultiModalDataset(X_val_w, y_val)
            test_dataset = MultiModalDataset(X_test_w, y_test)
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(
                model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
            )
            
            # Compile fold result
            fold_result = {
                'fold': fold_idx,
                'method': method_name,
                'num_train': len(X_train),
                'num_val': len(X_val),
                'num_test': len(X_test),
                'val_acc': eval_metrics['val_acc'],
                'val_loss': eval_metrics['val_loss'],
                'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'],
                'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'],
                'test_recall_macro': eval_metrics['test_recall_macro'],
                'modality_weights': modality_weights,
                'weight_optimization_time': weight_results['execution_time'],
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                'model_params': eval_metrics['model_params'],
                'model_size_mb': eval_metrics['model_size_mb'],
                'original_dataset_size_mb': orig_dataset_size_mb,
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                'convergence': weight_results.get('convergence', None),
                'best_fitness': weight_results.get('best_fitness', None),
                'test_preds': eval_metrics['test_preds'],
                'test_true': eval_metrics['test_true'],
            }
            
            master_results[method_name][fold_idx] = fold_result
            
            # Save per-fold result
            fold_dir = RESULTS_ROOT / method_name
            fold_save = {k: v for k, v in fold_result.items()
                         if k not in ['test_preds', 'test_true']}
            fold_save_clean = {}
            for k, v in fold_save.items():
                if isinstance(v, np.ndarray):
                    fold_save_clean[k] = v.tolist()
                elif isinstance(v, (np.floating, np.integer)):
                    fold_save_clean[k] = float(v)
                else:
                    fold_save_clean[k] = v
            
            with open(fold_dir / f"fold_{fold_idx+1}.json", 'w') as f:
                json_lib.dump(fold_save_clean, f, indent=2, default=str)
            
            # Save weights as npy for easy loading
            np.save(fold_dir / f"fold_{fold_idx+1}_weights.npy", modality_weights)
            
            w_str = ', '.join([f"{MODALITY_NAMES[i]}={modality_weights[i]:.3f}" for i in range(N_MODALITIES)])
            print(f"    Weights: [{w_str}]")
            print(f"    Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Model: {eval_metrics['model_size_mb']:.3f}MB")
            
            del model, train_dataset, val_dataset, test_dataset
            del train_loader, val_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()
    
    print(f"\n{'=' * 80}")
    print("ALL WEIGHTED FUSION EXPERIMENTS COMPLETED")
    print(f"{'=' * 80}")
    
    return master_results


print("Main experiment runner defined")

Main experiment runner defined


## Run All Experiments

In [20]:
master_results = run_weighted_fusion_all_folds()

WEIGHTED EARLY FUSION: MODALITY WEIGHT OPTIMIZATION
Methods: baseline + 14 metaheuristics = 15 total
Folds: 5

################################################################################
# FOLD 1/5
################################################################################
  Train: 681, Val: 198, Test: 286

  --- BASELINE ---
    Weights: [depth=1.000, sensor=1.000, skeleton=1.000]
    Test Acc: 96.15%, F1: 96.16%, Model: 5.444MB

  --- WEIGHTED_BAT ---
    Running weighted_BAT...
      Search dim: 3, Pop: 20, Iter: 30
BAT is optimizing  "fitness"
['At iteration 0 the best fitness is 0.0695122577516096']
['At iteration 1 the best fitness is 0.062169307975896766']
['At iteration 2 the best fitness is 0.062169307975896766']
['At iteration 3 the best fitness is 0.062169307975896766']
['At iteration 4 the best fitness is 0.05838197303403701']
['At iteration 5 the best fitness is 0.05393411894328892']
['At iteration 6 the best fitness is 0.05393411894328892']
['At iteration 7 the 

## Build Per-Fold Results CSV

In [21]:
rows = []
for method_name in ALL_METHODS:
    for fold_idx in range(N_FOLDS):
        if fold_idx not in master_results[method_name]:
            continue
        r = master_results[method_name][fold_idx]
        
        row = {
            'Method': method_name,
            'Fold': fold_idx + 1,
            'Test Accuracy (%)': r['test_acc'] * 100,
            'Val Accuracy (%)': r['val_acc'] * 100,
            'Val Loss': r.get('val_loss', float('inf')),
            'F1 Macro (%)': r['test_f1_macro'] * 100,
            'F1 Weighted (%)': r['test_f1_weighted'] * 100,
            'Precision Macro (%)': r['test_precision_macro'] * 100,
            'Recall Macro (%)': r['test_recall_macro'] * 100,
            'Weight Opt Time (s)': r['weight_optimization_time'],
            'Train Time (s)': r['train_time_sec'],
            'Total Time (s)': r['total_time_sec'],
            'Model Params': r['model_params'],
            'Model Size (MB)': r['model_size_mb'],
            'GPU Peak (MB)': r['gpu_mem_peak_mb'],
        }
        
        # Add per-modality weights
        mod_w = r['modality_weights']
        for i, mod_name in enumerate(MODALITY_NAMES):
            row[f'w_{mod_name}'] = mod_w[i]
        
        rows.append(row)

df_all = pd.DataFrame(rows).round(4)
df_all.to_csv(RESULTS_ROOT / "all_results_per_fold.csv", index=False)
print(f"Saved: {RESULTS_ROOT / 'all_results_per_fold.csv'}")
print(f"Shape: {df_all.shape}")
print(df_all.head(20).to_string())

Saved: results_weighted_fusion/all_results_per_fold.csv
Shape: (75, 18)
          Method  Fold  Test Accuracy (%)  Val Accuracy (%)  Val Loss  F1 Macro (%)  F1 Weighted (%)  Precision Macro (%)  Recall Macro (%)  Weight Opt Time (s)  Train Time (s)  Total Time (s)  Model Params  Model Size (MB)  GPU Peak (MB)  w_depth  w_sensor  w_skeleton
0       baseline     1            96.1538           98.4848    0.1460       96.1599          96.1599              96.9162           96.1538               0.0000          8.8853          8.8853       1425686           5.4444         50.209   1.0000    1.0000      1.0000
1       baseline     2            92.1162           98.4848    0.0591       92.1119          92.1010              93.2284           92.1074               0.0000          8.9223          8.9223       1425686           5.4444         50.209   1.0000    1.0000      1.0000
2       baseline     3            98.1818           99.4949    0.0259       98.1704          98.1704              98.4

## Summary Table

In [22]:
summary_rows = []
for method_name in ALL_METHODS:
    method_df = df_all[df_all['Method'] == method_name]
    if len(method_df) == 0:
        continue
    
    row = {
        'Method': method_name,
        'Mean Test Acc (%)': method_df['Test Accuracy (%)'].mean(),
        'Std Test Acc (%)': method_df['Test Accuracy (%)'].std(),
        'Mean F1 Macro (%)': method_df['F1 Macro (%)'].mean(),
        'Std F1 Macro (%)': method_df['F1 Macro (%)'].std(),
        'Mean Weight Opt Time (s)': method_df['Weight Opt Time (s)'].mean(),
        'Mean Train Time (s)': method_df['Train Time (s)'].mean(),
        'Mean Total Time (s)': method_df['Total Time (s)'].mean(),
        'Mean GPU Peak (MB)': method_df['GPU Peak (MB)'].mean(),
    }
    
    for mod_name in MODALITY_NAMES:
        row[f'Mean w_{mod_name}'] = method_df[f'w_{mod_name}'].mean()
        row[f'Std w_{mod_name}'] = method_df[f'w_{mod_name}'].std()
    
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(2)
df_summary = df_summary.sort_values('Mean Test Acc (%)', ascending=False)
df_summary.to_csv(RESULTS_ROOT / "summary_all_methods.csv", index=False)
print("\nSUMMARY TABLE (mean across folds):")
print(df_summary.to_string(index=False))


SUMMARY TABLE (mean across folds):
       Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Std F1 Macro (%)  Mean Weight Opt Time (s)  Mean Train Time (s)  Mean Total Time (s)  Mean GPU Peak (MB)  Mean w_depth  Std w_depth  Mean w_sensor  Std w_sensor  Mean w_skeleton  Std w_skeleton
 weighted_SCA              95.12              2.11              95.07              2.12                    240.87                 9.03               249.90               50.21          0.01         0.02           0.75          0.27             0.64            0.42
 weighted_BAT              95.03              1.78              94.85              1.91                    249.20                 9.01               258.21               50.21          0.05         0.07           0.62          0.27             0.49            0.36
     baseline              94.74              2.73              94.53              2.95                      0.00                 9.07                 9.07              

## Select Best Weights Per Fold

For each fold, pick the metaheuristic whose weighted fusion achieved the **highest val accuracy**.
Save these best weights so File 2 can load and apply them.

In [23]:
best_weights_per_fold = {}  # {fold_idx: {'weights': array, 'method': str, 'val_loss': float, ...}}

weighted_methods = [m for m in ALL_METHODS if m != 'baseline']

print("=" * 80)
print("BEST WEIGHTED METHOD PER FOLD (selected by lowest val loss)")
print("=" * 80)

for fold_idx in range(N_FOLDS):
    best_val_loss = float('inf')
    best_method = None
    best_weights = None
    best_test_acc = None
    best_val_acc = None
    
    for method_name in weighted_methods:
        if fold_idx in master_results[method_name]:
            r = master_results[method_name][fold_idx]
            if r['val_loss'] < best_val_loss:
                best_val_loss = r['val_loss']
                best_method = method_name
                best_weights = r['modality_weights'].copy()
                best_test_acc = r['test_acc']
                best_val_acc = r['val_acc']
    
    best_weights_per_fold[fold_idx] = {
        'weights': best_weights,
        'method': best_method,
        'val_loss': best_val_loss,
        'val_acc': best_val_acc,
        'test_acc': best_test_acc,
    }
    
    w_str = ', '.join([f"{MODALITY_NAMES[i]}={best_weights[i]:.3f}" for i in range(N_MODALITIES)])
    print(f"  Fold {fold_idx+1}: {best_method} | Val loss: {best_val_loss:.6f} | "
          f"Val acc: {best_val_acc*100:.2f}% | Test: {best_test_acc*100:.2f}% | Weights: [{w_str}]")

# Compare with baseline
print(f"\n  {'---' * 20}")
print(f"  Baseline comparison:")
for fold_idx in range(N_FOLDS):
    if fold_idx in master_results['baseline']:
        bl = master_results['baseline'][fold_idx]
        bw = best_weights_per_fold[fold_idx]
        diff = bw['test_acc'] - bl['test_acc']
        print(f"    Fold {fold_idx+1}: Baseline={bl['test_acc']*100:.2f}%, "
              f"Best weighted={bw['test_acc']*100:.2f}%, Diff={diff*100:+.2f}%")

# Save for File 2
save_obj = {}
for fold_idx, info in best_weights_per_fold.items():
    save_obj[fold_idx] = {
        'weights': info['weights'],
        'method': info['method'],
        'val_loss': info['val_loss'],
        'val_acc': info['val_acc'],
        'test_acc': info['test_acc'],
    }

joblib.dump(save_obj, RESULTS_ROOT / "best_weights_per_fold.pkl")
print(f"\nSaved: {RESULTS_ROOT / 'best_weights_per_fold.pkl'}")
print("This file will be loaded by File 2 for weighted feature selection.")


BEST WEIGHTED METHOD PER FOLD (selected by lowest val loss)
  Fold 1: weighted_HHO | Val loss: 0.028708 | Val acc: 99.49% | Test: 95.45% | Weights: [depth=0.004, sensor=0.014, skeleton=0.020]
  Fold 2: weighted_CS | Val loss: 0.030682 | Val acc: 99.49% | Test: 92.12% | Weights: [depth=0.047, sensor=0.529, skeleton=0.555]
  Fold 3: weighted_GWO | Val loss: 0.022584 | Val acc: 99.49% | Test: 97.27% | Weights: [depth=0.039, sensor=1.000, skeleton=0.895]
  Fold 4: weighted_GA | Val loss: 0.014666 | Val acc: 100.00% | Test: 90.91% | Weights: [depth=0.164, sensor=0.978, skeleton=0.580]
  Fold 5: weighted_SCA | Val loss: 0.110025 | Val acc: 96.68% | Test: 95.45% | Weights: [depth=0.016, sensor=0.499, skeleton=0.070]

  ------------------------------------------------------------
  Baseline comparison:
    Fold 1: Baseline=96.15%, Best weighted=95.45%, Diff=-0.70%
    Fold 2: Baseline=92.12%, Best weighted=92.12%, Diff=+0.00%
    Fold 3: Baseline=98.18%, Best weighted=97.27%, Diff=-0.91%
    F

## Visualizations

In [24]:
# ============================================================================
# PLOT 1: Test Accuracy comparison (box plot)
# ============================================================================
fig, ax = plt.subplots(figsize=(18, 7))
order = df_summary['Method'].tolist()
sns.boxplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, palette='Set2')
sns.stripplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax,
              color='black', alpha=0.5, size=4)
ax.set_title('Weighted Early Fusion: Test Accuracy Comparison', fontsize=14)
ax.set_ylabel('Test Accuracy (%)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'accuracy_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [25]:
# ============================================================================
# PLOT 2: Learned modality weights heatmap (all methods x folds)
# ============================================================================
for mod_name in MODALITY_NAMES:
    fig, ax = plt.subplots(figsize=(12, 8))
    pivot = df_all[df_all['Method'] != 'baseline'].pivot_table(
        values=f'w_{mod_name}', index='Method', columns='Fold'
    )
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
                vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Weight'})
    ax.set_title(f'Learned Weight for {mod_name.upper()} Modality', fontsize=14)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'weights_heatmap_{mod_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [26]:
# ============================================================================
# PLOT 3: Average modality weights bar chart (all optimizers)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))

weight_methods = [m for m in ALL_METHODS if m != 'baseline']
x = np.arange(len(weight_methods))
width = 0.18
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i, mod_name in enumerate(MODALITY_NAMES):
    means, stds = [], []
    for method in weight_methods:
        mdf = df_all[df_all['Method'] == method]
        means.append(mdf[f'w_{mod_name}'].mean())
        stds.append(mdf[f'w_{mod_name}'].std())
    ax.bar(x + i * width, means, width, yerr=stds, capsize=2,
           label=mod_name, color=colors[i], alpha=0.8)

ax.set_ylabel('Weight', fontsize=12)
ax.set_title('Average Learned Modality Weights by Optimizer', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([m.replace('weighted_', '') for m in weight_methods], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'avg_modality_weights_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [27]:
# ============================================================================
# PLOT 4: Improvement over baseline
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 6))

baseline_mean = df_summary[df_summary['Method'] == 'baseline']['Mean Test Acc (%)'].values[0]

improvements = []
method_labels = []
for _, row in df_summary.iterrows():
    if row['Method'] == 'baseline':
        continue
    improvements.append(row['Mean Test Acc (%)'] - baseline_mean)
    method_labels.append(row['Method'].replace('weighted_', ''))

colors_imp = ['green' if x >= 0 else 'red' for x in improvements]
bars = ax.bar(range(len(improvements)), improvements, color=colors_imp, alpha=0.7)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(range(len(improvements)))
ax.set_xticklabels(method_labels, rotation=45, ha='right')
ax.set_ylabel('Accuracy Improvement over Baseline (%)', fontsize=12)
ax.set_title('Weighted Fusion vs Simple Concatenation', fontsize=14)

for bar, val in zip(bars, improvements):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:+.2f}%', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'improvement_over_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [28]:
# ============================================================================
# PLOT 5: Convergence curves
# ============================================================================
fig, ax = plt.subplots(figsize=(14, 8))

for method in weight_methods:
    all_conv = []
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            conv = master_results[method][fold_idx].get('convergence', None)
            if conv is not None and len(conv) > 0:
                all_conv.append(conv)
    
    if len(all_conv) > 0:
        min_len = min(len(c) for c in all_conv)
        all_conv_arr = np.array([c[:min_len] for c in all_conv])
        mean_conv = np.mean(all_conv_arr, axis=0)
        ax.plot(range(1, min_len + 1), mean_conv,
                label=method.replace('weighted_', ''), linewidth=2)

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Fitness (lower is better)', fontsize=12)
ax.set_title('Weight Optimization Convergence (averaged across folds)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'convergence_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [29]:
# ============================================================================
# PLOT 6: Best weights per fold (the ones that will be used in File 2)
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 6))

fold_labels = [f"Fold {i+1}\n({best_weights_per_fold[i]['method'].replace('weighted_', '')})" 
               for i in range(N_FOLDS)]
x = np.arange(N_FOLDS)
width = 0.18
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i, mod_name in enumerate(MODALITY_NAMES):
    weights = [best_weights_per_fold[fold_idx]['weights'][i] for fold_idx in range(N_FOLDS)]
    ax.bar(x + i * width, weights, width, label=mod_name, color=colors[i], alpha=0.8)

ax.set_ylabel('Weight', fontsize=12)
ax.set_title('Best Learned Modality Weights Per Fold (used in File 2)', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(fold_labels, fontsize=9)
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'best_weights_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Significance

In [30]:
from scipy.stats import wilcoxon, ttest_rel

print("=" * 80)
print("STATISTICAL SIGNIFICANCE: each weighted method vs baseline")
print("=" * 80)

baseline_accs = []
for fold_idx in range(N_FOLDS):
    if fold_idx in master_results['baseline']:
        baseline_accs.append(master_results['baseline'][fold_idx]['test_acc'] * 100)

stat_rows = []
for method in ALL_METHODS:
    if method == 'baseline':
        continue
    
    method_accs = []
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            method_accs.append(master_results[method][fold_idx]['test_acc'] * 100)
    
    if len(method_accs) == len(baseline_accs) and len(method_accs) >= 2:
        try:
            t_stat, t_pval = ttest_rel(method_accs, baseline_accs)
        except:
            t_stat, t_pval = np.nan, np.nan
        try:
            w_stat, w_pval = wilcoxon(np.array(method_accs) - np.array(baseline_accs))
        except:
            w_stat, w_pval = np.nan, np.nan
        
        diff = np.mean(method_accs) - np.mean(baseline_accs)
        sig = '***' if t_pval < 0.001 else '**' if t_pval < 0.01 else '*' if t_pval < 0.05 else ''
        
        stat_rows.append({
            'Method': method,
            'Mean Diff (%)': diff,
            'Paired t p-val': t_pval,
            'Wilcoxon p-val': w_pval,
            'Sig': sig
        })
        
        print(f"  {method:25s}: diff={diff:+.2f}%, t-test p={t_pval:.4f}, "
              f"wilcoxon p={w_pval:.4f} {sig}")

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(RESULTS_ROOT / "statistical_tests.csv", index=False)

STATISTICAL SIGNIFICANCE: each weighted method vs baseline
  weighted_BAT             : diff=+0.29%, t-test p=0.7133, wilcoxon p=0.8125 
  weighted_CS              : diff=-2.65%, t-test p=0.0673, wilcoxon p=0.1250 
  weighted_DE              : diff=-0.89%, t-test p=0.3605, wilcoxon p=0.4375 
  weighted_FFA             : diff=-0.14%, t-test p=0.8168, wilcoxon p=0.8750 
  weighted_GA              : diff=-0.12%, t-test p=0.7538, wilcoxon p=0.8750 
  weighted_GWO             : diff=-0.86%, t-test p=0.5455, wilcoxon p=0.6250 
  weighted_HHO             : diff=-0.44%, t-test p=0.2542, wilcoxon p=0.2500 
  weighted_JAYA            : diff=-0.79%, t-test p=0.3260, wilcoxon p=0.3125 
  weighted_MFO             : diff=-0.88%, t-test p=0.1712, wilcoxon p=0.2500 
  weighted_MVO             : diff=-0.36%, t-test p=0.7784, wilcoxon p=0.8125 
  weighted_PSO             : diff=-1.38%, t-test p=0.3914, wilcoxon p=0.6250 
  weighted_SCA             : diff=+0.37%, t-test p=0.6341, wilcoxon p=0.6250 
  wei

## Final Summary

In [31]:
print("=" * 80)
print("FILE 1 COMPLETE — WEIGHTED EARLY FUSION")
print("=" * 80)

print(f"\nOutput directory: {RESULTS_ROOT}")
print(f"\nCSV files:")
for p in sorted(RESULTS_ROOT.glob('*.csv')):
    print(f"  {p.name}")
print(f"\nPlots:")
for p in sorted(PLOTS_DIR.glob('*.png')):
    print(f"  {p.name}")

print(f"\n{'=' * 80}")
print("RESULTS (sorted by mean test accuracy):")
print("=" * 80)
for _, row in df_summary.iterrows():
    w_str = ' | '.join([f"{m}={row[f'Mean w_{m}']:.3f}" for m in MODALITY_NAMES])
    print(f"  {row['Method']:25s} | Acc: {row['Mean Test Acc (%)']:6.2f}% "
          f"(+/-{row['Std Test Acc (%)']:.2f}) | {w_str}")

print(f"\n{'=' * 80}")
print("BEST WEIGHTS PER FOLD (to be used in File 2):")
print("=" * 80)
for fold_idx in range(N_FOLDS):
    info = best_weights_per_fold[fold_idx]
    w_str = ', '.join([f"{MODALITY_NAMES[i]}={info['weights'][i]:.3f}" for i in range(N_MODALITIES)])
    print(f"  Fold {fold_idx+1}: {info['method']:25s} | Val loss: {info['val_loss']:.6f} | Val acc: {info['val_acc']*100:.2f}% | [{w_str}]")

print(f"\nKey output for File 2: {RESULTS_ROOT / 'best_weights_per_fold.pkl'}")

FILE 1 COMPLETE — WEIGHTED EARLY FUSION

Output directory: results_weighted_fusion

CSV files:
  all_results_per_fold.csv
  statistical_tests.csv
  summary_all_methods.csv

Plots:
  accuracy_boxplot.png
  avg_modality_weights_bar.png
  best_weights_per_fold.png
  convergence_curves.png
  improvement_over_baseline.png
  weights_heatmap_depth.png
  weights_heatmap_sensor.png
  weights_heatmap_skeleton.png

RESULTS (sorted by mean test accuracy):
  weighted_SCA              | Acc:  95.12% (+/-2.11) | depth=0.010 | sensor=0.750 | skeleton=0.640
  weighted_BAT              | Acc:  95.03% (+/-1.78) | depth=0.050 | sensor=0.620 | skeleton=0.490
  baseline                  | Acc:  94.74% (+/-2.73) | depth=1.000 | sensor=1.000 | skeleton=1.000
  weighted_GA               | Acc:  94.62% (+/-2.62) | depth=0.160 | sensor=0.790 | skeleton=0.610
  weighted_FFA              | Acc:  94.61% (+/-2.85) | depth=0.080 | sensor=0.720 | skeleton=0.570
  weighted_MVO              | Acc:  94.38% (+/-1.95) | de